# BirdCLEF+ 2026 — Training Notebook

このノートブックは BirdCLEF 2026 コンペの **学習用** ノートブックです。

## 使い方
1. このリポジトリ全体を Kaggle Dataset として `birdclef-2026-code` という名前でアップロード
2. 本ノートブックに以下を追加:
   - Competition: `birdclef-2026`
   - Dataset: `birdclef-2026-code` (自分のリポジトリ)
3. GPU アクセラレータを有効化して実行
4. 学習完了後、`/kaggle/working/` に保存された `.pth` ファイルを Kaggle Dataset としてアップロード

## 学習設定 (improved.yaml)
- Model: `tf_efficientnet_b3_ns` + GeM Pooling
- Loss: Focal Loss (γ=2.0, α=0.25)
- Augmentation: SpecAugment + Pink Noise + Mixup (α=0.4)
- Secondary labels: soft weight 0.5
- 5-fold cross-validation
- Metric: Padded cmAP (padding_factor=5)

In [ ]:
import sys, os

# リポジトリをパスに追加（Kaggle Dataset として追加した場合）
REPO_PATH = '/kaggle/input/birdclef-2026-code'
if os.path.exists(REPO_PATH) and REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# 確認
print('Python:', sys.version)
print('Working dir:', os.getcwd())
print('Repo found:', os.path.exists(REPO_PATH))

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import pandas as pd

COMP_DATA = '/kaggle/input/birdclef-2026'
metadata = pd.read_csv(f'{COMP_DATA}/train_metadata.csv')
print('Train metadata shape:', metadata.shape)
print('Columns:', metadata.columns.tolist())
print('\nSample rows:')
metadata.head(3)

In [ ]:
import matplotlib.pyplot as plt

# 種別サンプル数の分布
class_counts = metadata['primary_label'].value_counts()
print(f'総クラス数: {len(class_counts)}')
print(f'最小サンプル数: {class_counts.min()}')
print(f'最大サンプル数: {class_counts.max()}')
print(f'中央値: {class_counts.median():.0f}')
print(f'\n最少上位10種:')
print(class_counts.tail(10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
class_counts.hist(bins=50, ax=ax1)
ax1.set_title('クラス別サンプル数の分布')
ax1.set_xlabel('サンプル数')
ax1.set_ylabel('クラス数')

class_counts.head(30).plot(kind='bar', ax=ax2)
ax2.set_title('上位30クラスのサンプル数')
ax2.set_xlabel('種')
ax2.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
import yaml

# improved.yaml の内容をインラインで定義
# （リポジトリから読む場合は load_config を使う）
CONFIG = {
    # Model
    'model_name': 'tf_efficientnet_b3_ns',
    'pretrained': True,
    'use_gem_pooling': True,
    'gem_p': 3.0,

    # Training
    'num_epochs': 30,
    'batch_size': 32,
    'learning_rate_backbone': 1e-4,
    'learning_rate_head': 1e-3,
    'weight_decay': 1e-6,
    'n_folds': 5,
    'seed': 42,
    'fp16': True,
    'early_stopping_patience': 7,
    'grad_clip': 1.0,

    # Audio
    'sample_rate': 32000,
    'duration': 5,
    'n_mels': 224,
    'n_fft': 2048,
    'hop_length': 256,
    'fmin': 50,
    'fmax': 16000,

    # SpecAugment
    'freq_mask_param': 27,
    'time_mask_param': 62,
    'num_freq_masks': 2,
    'num_time_masks': 2,

    # Mixup
    'mixup_alpha': 0.4,

    # Secondary labels
    'secondary_label_weight': 0.5,

    # Focal Loss
    'use_focal_loss': True,
    'focal_gamma': 2.0,
    'focal_alpha': 0.25,

    # Pink noise
    'pink_noise_prob': 0.3,
    'pink_noise_snr_db_min': 5.0,
    'pink_noise_snr_db_max': 20.0,

    # Inference
    'inference_overlap': 2.5,
    'inference_tta': False,

    # Paths
    'train_audio_dir': f'{COMP_DATA}/train_audio',
    'train_metadata': f'{COMP_DATA}/train_metadata.csv',
    'taxonomy': f'{COMP_DATA}/taxonomy.csv',
    'test_soundscapes_dir': f'{COMP_DATA}/test_soundscapes',
    'sample_submission': f'{COMP_DATA}/sample_submission.csv',
    'output_dir': '/kaggle/working',
}

# YAML として保存（src/train.py から読めるように）
os.makedirs('/kaggle/working', exist_ok=True)
config_path = '/kaggle/working/config_train.yaml'
with open(config_path, 'w') as f:
    yaml.dump(CONFIG, f)
print(f'Config saved to: {config_path}')

In [ ]:
# 全 fold の学習を実行
# ※ 1 fold だけ試す場合は --fold 0 を追加
!python {REPO_PATH}/src/train.py --config {config_path}

In [ ]:
# 学習済みチェックポイントの確認
import glob
checkpoints = sorted(glob.glob('/kaggle/working/fold*.pth'))
print(f'保存されたチェックポイント: {len(checkpoints)} 個')
for cp in checkpoints:
    ckpt = torch.load(cp, map_location='cpu')
    print(f'  {os.path.basename(cp)} — val_cmap={ckpt["val_score"]:.4f} (epoch {ckpt["epoch"]})')

## 次のステップ

### Step 1: モデルをデータセットとしてアップロード
1. Kaggle の「Output」タブから `/kaggle/working/` の内容を確認
2. `fold0_best.pth` ～ `fold4_best.pth` と `label_encoder.pkl` を含む Dataset を作成
3. Dataset 名を `birdclef-2026-models` とする

### Step 2: 推論ノートブックで提出
1. `inference_kaggle.ipynb` を新しい Kaggle Notebook として作成
2. 以下のデータセットを追加:
   - `birdclef-2026` (competition)
   - `birdclef-2026-models` (自分のモデル)
   - `birdclef-2026-code` (このリポジトリ)
3. Accelerator: **None (CPU)** で実行
4. 提出

### Step 3: 精度向上 (→ README.md 参照)

## オプション: Pseudo-Labeling (Round 2)

ラベルなしサウンドスケープを使ってモデルを強化できます。
Round 1 学習後に以下を実行してください。

In [ ]:
# ラベルなしサウンドスケープが存在する場合のみ実行
UNLABELED_DIR = f'{COMP_DATA}/unlabeled_soundscapes'
if os.path.isdir(UNLABELED_DIR):
    !python {REPO_PATH}/src/pseudo_label.py \
        --config {config_path} \
        --soundscapes_dir {UNLABELED_DIR} \
        --output_csv /kaggle/working/pseudo_labels.csv \
        --threshold 0.5 \
        --auto_cap
    
    # Round 2: pseudo ラベルと合わせて再学習
    PSEUDO_CONFIG = CONFIG.copy()
    PSEUDO_CONFIG.update({
        'num_epochs': 15,
        'learning_rate_backbone': 5e-5,
        'learning_rate_head': 5e-4,
        'early_stopping_patience': 4,
        'pseudo_labels_path': '/kaggle/working/pseudo_labels.csv',
        'pseudo_mix_ratio': 0.3,
        'output_dir': '/kaggle/working/pseudo_round2',
    })
    pseudo_config_path = '/kaggle/working/config_pseudo.yaml'
    with open(pseudo_config_path, 'w') as f:
        yaml.dump(PSEUDO_CONFIG, f)
    
    !python {REPO_PATH}/src/train.py --config {pseudo_config_path}
else:
    print('unlabeled_soundscapes が見つかりません。スキップします。')